# Harbor Chatbot — LoRA Finetuning

This notebook finetunes SmolLM3-3B with QLoRA on substance abuse & mental health counseling conversations.

**Requirements:** Run this on a GPU runtime (Runtime → Change runtime type → T4 GPU).

### 1. Install dependencies

In [ ]:
!pip install -q unsloth trl datasets transformers accelerate huggingface-hub

### 2. Configuration

Set your HF token and repo ID here. Upload `examples.jsonl` to the Colab file browser (left sidebar).

In [ ]:
from google.colab import userdata, files
import os

# --- Configuration ---
BASE_MODEL = "HuggingFaceTB/SmolLM3-3B"
OUTPUT_DIR = "./harbor-lora"
HF_REPO_ID = "amitashukla/harbor-smollm3-lora"

# Option 1: Set HF_TOKEN in Colab Secrets (Key icon in left sidebar) — recommended
# Option 2: Paste your token directly (not recommended for shared notebooks)
HF_TOKEN = userdata.get("HF_TOKEN")

# Upload examples.jsonl if not already present
if not os.path.exists("examples.jsonl"):
    print("Upload examples.jsonl:")
    uploaded = files.upload()
    print(f"Uploaded: {list(uploaded.keys())}")

### 3. Load base model with 4-bit QLoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    load_in_4bit=True,
    token=HF_TOKEN,
)

### 4. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
)

### 5. Load and format dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="examples.jsonl", split="train")

# Train/validation split (85/15)
split = dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

# Format using the model's native chat template
ROLE_MAP = {"human": "user", "gpt": "assistant"}

def format_conversation(example):
    messages = [
        {"role": ROLE_MAP[turn["from"]], "content": turn["value"]}
        for turn in example["conversations"]
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_dataset = train_dataset.map(format_conversation)
eval_dataset = eval_dataset.map(format_conversation)

print(f"Train: {len(train_dataset)} examples, Eval: {len(eval_dataset)} examples")
print(f"\nSample formatted text:\n{train_dataset[0]['text'][:500]}...")

### 6. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        eval_strategy="epoch",
        output_dir=OUTPUT_DIR,
        save_strategy="epoch",
    ),
)

trainer.train()

### 7. Push to Hugging Face Hub

After this completes, set `MY_MODEL` in `src/config.py` to your `HF_REPO_ID` value.

In [ ]:
print(f"Pushing adapter to Hugging Face Hub: {HF_REPO_ID}")
model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
print(f'Done! Set MY_MODEL = "{HF_REPO_ID}" in src/config.py to use it.')

### 8. Quick test (optional)

Sanity-check the finetuned model before leaving Colab.

In [ ]:
FastLanguageModel.for_inference(model)

test_message = [{"role": "user", "content": "I think I might have a problem with drinking. What should I do?"}]
inputs = tokenizer.apply_chat_template(test_message, return_tensors="pt", add_generation_prompt=True).to("cuda")

output = model.generate(inputs, max_new_tokens=256, temperature=0.7)
print(tokenizer.decode(output[0], skip_special_tokens=True))